# 04 — Model Training
## SustAInable | Neighborhood Heat Illness Risk Prediction

---

**Purpose:** Trains the XGBoost supervised classification model that predicts which ZIP codes will experience elevated heat illness during an approaching extreme heat event.

**Inputs:**
- `data/processed/sustainable_labeled.csv` ← from Notebook 03

**Outputs:**
- `outputs/sustainable_model.json` — trained XGBoost model
- `outputs/sustainable_model_metadata.json` — threshold, features, training config
- `outputs/04a_cv_results.png` — cross-validation performance chart
- `outputs/04b_threshold_curve.png` — precision/recall vs. threshold

**The core design principle:** A false negative — missing a ZIP code that will experience elevated heat illness — is categorically worse than a false positive. Every technical decision in this notebook is made with that asymmetry in mind.

**Author:** Nico Meyering, MPA | Equitech Futures Civic Tech Institute, CTI 2026

---

## Step 0: Imports and Dependency Check

This notebook requires `xgboost`, `scikit-learn`, and `imbalanced-learn`. This cell checks for all three and tells you exactly what to install if anything is missing.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import json
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 50)

# ── Dependency checks ──────────────────────────────────────────────────────────
missing = []

try:
    import xgboost as xgb
    print(f"✅ xgboost        {xgb.__version__}")
except ImportError:
    missing.append("xgboost")
    print("❌ xgboost        — not installed")

try:
    import sklearn
    from sklearn.model_selection import StratifiedKFold, cross_validate
    from sklearn.metrics import (
        roc_auc_score, recall_score, precision_score,
        f1_score, confusion_matrix, precision_recall_curve,
        average_precision_score
    )
    from sklearn.preprocessing import StandardScaler
    print(f"✅ scikit-learn   {sklearn.__version__}")
except ImportError:
    missing.append("scikit-learn")
    print("❌ scikit-learn   — not installed")

try:
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline
    import imblearn
    print(f"✅ imbalanced-learn {imblearn.__version__}")
    SMOTE_AVAILABLE = True
except ImportError:
    missing.append("imbalanced-learn")
    print("⚠️  imbalanced-learn — not installed (SMOTE will be skipped)")
    SMOTE_AVAILABLE = False

if missing:
    print()
    print("Install missing packages with:")
    install_str = " ".join(missing)
    print(f"  pip install {install_str}")
    print()
    print("Then restart the kernel and re-run this cell.")
else:
    print()
    print("✅ All dependencies satisfied. Ready to train.")

## Step 1: File Paths

In [ ]:
DATA_PROCESSED = Path("../data/processed")
OUTPUTS        = Path("../outputs")
OUTPUTS.mkdir(parents=True, exist_ok=True)

LABELED_PATH   = DATA_PROCESSED / "sustainable_labeled.csv"
MODEL_PATH     = OUTPUTS / "sustainable_model.json"
METADATA_PATH  = OUTPUTS / "sustainable_model_metadata.json"

print(f"Labeled data: {'✅  found' if LABELED_PATH.exists() else '🔴  NOT FOUND — run Notebook 03 first'}")
print(f"Outputs dir:  {OUTPUTS}")

## Step 2: Load Labeled Dataset

Loads the output of Notebook 03 and gives a full accounting of what we have before any data transformations happen.

In [ ]:
labeled_df = None

if LABELED_PATH.exists():
    labeled_df = pd.read_csv(LABELED_PATH, dtype={'ZCTA': str})
    labeled_df['ZCTA'] = labeled_df['ZCTA'].str.zfill(5)

    # Detect proxy label mode
    IS_PROXY = 'label_is_proxy' in labeled_df.columns and labeled_df['label_is_proxy'].max() == 1

    print("=" * 58)
    print("LABELED DATASET SUMMARY")
    print("=" * 58)
    print(f"Total rows:       {len(labeled_df):,}  (ZCTA × event observations)")
    print(f"Total columns:    {labeled_df.shape[1]}")
    print(f"Label mode:       {'⚠️  PROXY (structurally derived)' if IS_PROXY else '✅  REAL (NSSP ED visit data)'}")
    print()

    train_df   = labeled_df[labeled_df['split'] == 'train']
    holdout_df = labeled_df[labeled_df['split'] == 'holdout']

    for name, df in [("Training", train_df), ("Holdout", holdout_df)]:
        vc  = df['label'].value_counts()
        pos = vc.get(1, 0)
        neg = vc.get(0, 0)
        rat = neg / pos if pos > 0 else float('inf')
        print(f"{name} set:")
        print(f"  Rows:       {len(df):,}")
        print(f"  Label = 0:  {neg:,}  ({neg/len(df)*100:.1f}%)")
        print(f"  Label = 1:  {pos:,}  ({pos/len(df)*100:.1f}%)")
        print(f"  Imbalance:  {rat:.1f}:1")
        print()

    if IS_PROXY:
        print("⚠️  PROXY LABEL REMINDER")
        print("   This model is trained on structurally derived labels.")
        print("   It is suitable for pipeline development and testing ONLY.")
        print("   Replace labels with NSSP data before any real deployment.")
else:
    print("🔴 Labeled dataset not found.")
    print(f"   Expected: {LABELED_PATH}")
    print("   Run Notebook 03 first.")

## Step 3: Build Feature Matrix (X) and Label Vector (Y)

We need to separate:
- **X** — the input features (everything the model learns from)
- **Y** — the label (what the model predicts)

We also need to drop columns that would leak information about the outcome or that aren't real features (like event IDs and dates).

In [ ]:
X_train = Y_train = X_holdout = Y_holdout = None
FEATURE_NAMES = []

# ── Columns to explicitly exclude from features ────────────────────────────────
NON_FEATURE_COLS = [
    'ZCTA', 'label', 'split', 'event_id', 'event_label',
    'start_date', 'end_date', 'duration_days', 'year',
    'label_is_proxy', 'vulnerability_score_used',
    'base_prob', 'event_visits', 'event_rate_per_10k',
    'baseline_rate_per_10k', 'population',
]

if labeled_df is not None:
    train_df   = labeled_df[labeled_df['split'] == 'train'].copy()
    holdout_df = labeled_df[labeled_df['split'] == 'holdout'].copy()

    # Identify feature columns
    FEATURE_NAMES = [
        c for c in labeled_df.columns
        if c not in NON_FEATURE_COLS
        and pd.api.types.is_numeric_dtype(labeled_df[c])
    ]

    # Build matrices
    X_train   = train_df[FEATURE_NAMES].copy()
    Y_train   = train_df['label'].copy()
    X_holdout = holdout_df[FEATURE_NAMES].copy()
    Y_holdout = holdout_df['label'].copy()

    # Final null check and fill
    null_train   = X_train.isnull().sum().sum()
    null_holdout = X_holdout.isnull().sum().sum()
    if null_train > 0:
        X_train = X_train.fillna(X_train.median())
        print(f"ℹ️  Filled {null_train} nulls in training features with median.")
    if null_holdout > 0:
        X_holdout = X_holdout.fillna(X_train.median())  # use TRAIN median for holdout
        print(f"ℹ️  Filled {null_holdout} nulls in holdout features with train median.")

    # Imbalance ratio for scale_pos_weight
    n_neg = (Y_train == 0).sum()
    n_pos = (Y_train == 1).sum()
    IMBALANCE_RATIO = round(n_neg / n_pos, 2) if n_pos > 0 else 1.0

    print(f"✅ Feature matrix ready.")
    print(f"   Features:          {len(FEATURE_NAMES)}")
    print(f"   Training rows:     {len(X_train):,}")
    print(f"   Holdout rows:      {len(X_holdout):,}")
    print(f"   Imbalance ratio:   {IMBALANCE_RATIO:.2f}:1  → scale_pos_weight")
    print()
    print(f"Feature columns ({len(FEATURE_NAMES)}):")
    for name_chunk in [FEATURE_NAMES[i:i+5] for i in range(0, len(FEATURE_NAMES), 5)]:
        print("  " + "  |  ".join(name_chunk))

## Step 4: SMOTE Oversampling

**What SMOTE does:** Instead of just duplicating minority-class rows, SMOTE (*Synthetic Minority Over-sampling Technique*) generates *new* synthetic examples by interpolating between existing positive-class rows. Think of it as teaching the model what "elevated heat illness neighborhoods" look like by showing it more examples, not just copies of the same ones.

**Why we use it:** At high imbalance ratios (>10:1), XGBoost can technically achieve high accuracy by predicting "0" for almost everything. SMOTE forces the model to actually learn what makes a ZCTA a 1.

**Important:** SMOTE is applied ONLY to the training set. The holdout set stays untouched — it represents real-world class distribution.

In [ ]:
X_train_resampled = X_train
Y_train_resampled = Y_train

if X_train is not None:
    if SMOTE_AVAILABLE:
        try:
            # k_neighbors must be < number of minority samples
            n_minority = (Y_train == 1).sum()
            k = min(5, n_minority - 1) if n_minority > 1 else 1

            smote = SMOTE(
                sampling_strategy=0.4,   # bring positives to 40% of negatives
                k_neighbors=k,
                random_state=42
            )
            X_train_resampled, Y_train_resampled = smote.fit_resample(X_train, Y_train)

            before_pos = (Y_train == 1).sum()
            after_pos  = (Y_train_resampled == 1).sum()
            before_neg = (Y_train == 0).sum()
            after_neg  = (Y_train_resampled == 0).sum()

            print("✅ SMOTE applied.")
            print(f"   Before resampling:  {before_neg:,} negative  |  {before_pos:,} positive")
            print(f"   After resampling:   {after_neg:,} negative  |  {after_pos:,} positive")
            new_ratio = after_neg / after_pos if after_pos > 0 else float('inf')
            print(f"   New ratio:          {new_ratio:.2f}:1")

            # Convert back to DataFrame to keep feature names
            X_train_resampled = pd.DataFrame(X_train_resampled, columns=FEATURE_NAMES)
            Y_train_resampled = pd.Series(Y_train_resampled, name='label')

        except Exception as e:
            print(f"⚠️  SMOTE failed: {e}")
            print("   Falling back to class weighting only (still valid).")
            X_train_resampled = X_train
            Y_train_resampled = Y_train
    else:
        print("ℹ️  SMOTE not available — using class weighting only.")
        print("   Install with: pip install imbalanced-learn")
        print("   Class weighting via scale_pos_weight is still effective.")

## Step 5: Model Configuration

Every hyperparameter here is chosen deliberately based on the SustAInable design spec. The comments explain the reasoning behind each one.

> If you want to experiment with different values, change them here and re-run Steps 5–8. The cross-validation in Step 6 will show you whether the change helped.

In [ ]:
# ── Core XGBoost parameters ────────────────────────────────────────────────────
XGB_PARAMS = {
    # ── Objective and class handling ──────────────────────────────────────────
    'objective':        'binary:logistic',
    'eval_metric':      ['aucpr', 'auc', 'logloss'],
    'scale_pos_weight': IMBALANCE_RATIO if X_train is not None else 5.0,
    # ^ Tells XGBoost how much to penalize missing a positive (elevated illness)

    # ── Tree structure ────────────────────────────────────────────────────────
    'max_depth':        5,
    # ^ Deeper trees learn more complex patterns but risk overfitting.
    #   5 is a conservative starting point for tabular civic data.
    'n_estimators':     400,
    # ^ Number of trees. More trees = better performance up to a point.
    #   Early stopping in Step 6 will find the optimal number automatically.
    'learning_rate':    0.05,
    # ^ Lower learning rate = slower but more stable learning.
    #   Pairs with more n_estimators.

    # ── Regularization (prevent overfitting) ─────────────────────────────────
    'subsample':        0.8,
    # ^ Each tree sees 80% of training rows — reduces overfitting.
    'colsample_bytree': 0.8,
    # ^ Each tree sees 80% of features — reduces overfitting.
    'min_child_weight': 5,
    # ^ Minimum observations in a leaf node. Higher = more conservative splits.
    'gamma':            0.1,
    # ^ Minimum loss reduction to make a split. Small regularization term.
    'reg_alpha':        0.1,   # L1 regularization
    'reg_lambda':       1.0,   # L2 regularization

    # ── Reproducibility and performance ──────────────────────────────────────
    'random_state':     42,
    'n_jobs':           -1,    # use all available CPU cores
    'tree_method':      'hist',  # fast histogram-based algorithm
    'verbosity':        0,
}

# ── Decision threshold ─────────────────────────────────────────────────────────
# Default XGBoost threshold is 0.50 (predict 1 if score >= 0.50).
# SustAInable lowers this to 0.30 to maximize recall.
# Asymmetric cost: false negatives (missed elevated ZCTAs) are worse than
# false positives (unnecessarily flagging a safe ZCTA).
DECISION_THRESHOLD = 0.30

print("XGBoost Configuration:")
print(f"  Objective:          {XGB_PARAMS['objective']}")
print(f"  Trees:              {XGB_PARAMS['n_estimators']}")
print(f"  Max depth:          {XGB_PARAMS['max_depth']}")
print(f"  Learning rate:      {XGB_PARAMS['learning_rate']}")
print(f"  scale_pos_weight:   {XGB_PARAMS['scale_pos_weight']:.2f}")
print(f"  Decision threshold: {DECISION_THRESHOLD}  (lowered from default 0.50)")
print(f"  Random state:       {XGB_PARAMS['random_state']}")

## Step 6: Stratified Cross-Validation

Before training the final model, we run 5-fold stratified cross-validation on the training set. This means the training data is split into 5 chunks — the model is trained on 4 chunks and tested on the 5th, rotating until every chunk has been the test set once.

**Why stratified?** "Stratified" means each fold keeps the same class ratio as the full training set — so we never accidentally train a fold with no positive examples.

**What we're measuring:**
- **AUC-ROC** — overall discrimination ability (target: ≥ 0.85)
- **Recall** — how often we catch elevated-illness ZCTAs (target: ≥ 0.80)
- **Precision** — of what we flag, how often are we right (floor: ≥ 0.40)
- **PR-AUC** — precision-recall area under curve (better metric for imbalanced data)

In [ ]:
cv_results_summary = None

if X_train_resampled is not None and 'xgb' in dir():
    print("Running 5-fold stratified cross-validation...")
    print("(This may take 1–3 minutes depending on your machine)\n")

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    fold_results = []

    for fold_idx, (train_idx, val_idx) in enumerate(
        skf.split(X_train_resampled, Y_train_resampled), start=1
    ):
        X_fold_train = X_train_resampled.iloc[train_idx]
        Y_fold_train = Y_train_resampled.iloc[train_idx]
        X_fold_val   = X_train_resampled.iloc[val_idx]
        Y_fold_val   = Y_train_resampled.iloc[val_idx]

        model_fold = xgb.XGBClassifier(**XGB_PARAMS)
        model_fold.fit(
            X_fold_train, Y_fold_train,
            eval_set=[(X_fold_val, Y_fold_val)],
            verbose=False
        )

        probs = model_fold.predict_proba(X_fold_val)[:, 1]
        preds = (probs >= DECISION_THRESHOLD).astype(int)

        auc_roc  = roc_auc_score(Y_fold_val, probs)
        pr_auc   = average_precision_score(Y_fold_val, probs)
        recall   = recall_score(Y_fold_val, preds, zero_division=0)
        precision= precision_score(Y_fold_val, preds, zero_division=0)
        f1       = f1_score(Y_fold_val, preds, zero_division=0)

        fold_results.append({
            'fold':       fold_idx,
            'auc_roc':    round(auc_roc, 4),
            'pr_auc':     round(pr_auc, 4),
            'recall':     round(recall, 4),
            'precision':  round(precision, 4),
            'f1':         round(f1, 4),
        })
        print(f"  Fold {fold_idx} | AUC-ROC: {auc_roc:.4f} | PR-AUC: {pr_auc:.4f} | "
              f"Recall: {recall:.4f} | Precision: {precision:.4f}")

    cv_df = pd.DataFrame(fold_results)
    means = cv_df.drop('fold', axis=1).mean().round(4)
    stds  = cv_df.drop('fold', axis=1).std().round(4)

    print()
    print("=" * 58)
    print("CROSS-VALIDATION SUMMARY (5-fold, threshold = {:.2f})".format(DECISION_THRESHOLD))
    print("=" * 58)
    metrics = ['auc_roc', 'pr_auc', 'recall', 'precision', 'f1']
    targets = {'auc_roc': 0.85, 'recall': 0.80, 'precision': 0.40}
    for m in metrics:
        target_str = f"  [target: ≥ {targets[m]}]" if m in targets else ""
        status = ""
        if m in targets:
            status = "✅" if means[m] >= targets[m] else "⚠️ "
        print(f"  {m:<12} {means[m]:.4f} ± {stds[m]:.4f}  {status}{target_str}")

    cv_results_summary = {'means': means.to_dict(), 'stds': stds.to_dict()}

    # ── CV visualization ───────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('SustAInable — 5-Fold Cross-Validation Results', fontsize=13, fontweight='bold')

    metric_colors = {
        'auc_roc':   '#3498db',
        'pr_auc':    '#9b59b6',
        'recall':    '#e74c3c',
        'precision': '#2ecc71',
        'f1':        '#f39c12'
    }

    # Per-fold lines
    for m, color in metric_colors.items():
        axes[0].plot(cv_df['fold'], cv_df[m], marker='o', label=m.upper().replace('_', '-'),
                     color=color, linewidth=2, markersize=7)
    axes[0].set_xlabel('Fold')
    axes[0].set_ylabel('Score')
    axes[0].set_title('Metrics per Fold')
    axes[0].legend(fontsize=9)
    axes[0].set_xticks([1, 2, 3, 4, 5])
    axes[0].set_ylim(0, 1.05)
    axes[0].axhline(0.85, color='#3498db', linestyle=':', alpha=0.5, linewidth=1)
    axes[0].axhline(0.80, color='#e74c3c', linestyle=':', alpha=0.5, linewidth=1)

    # Mean ± std bar
    mean_vals = [means[m] for m in metrics]
    std_vals  = [stds[m]  for m in metrics]
    bar_colors= [metric_colors[m] for m in metrics]
    x_pos = np.arange(len(metrics))
    bars  = axes[1].bar(x_pos, mean_vals, yerr=std_vals, color=bar_colors,
                        edgecolor='white', capsize=6, alpha=0.85)
    axes[1].set_xticks(x_pos)
    axes[1].set_xticklabels([m.upper().replace('_','-') for m in metrics], fontsize=9)
    axes[1].set_ylim(0, 1.05)
    axes[1].set_title('Mean ± Std Across Folds')
    axes[1].set_ylabel('Score')
    for bar, val in zip(bars, mean_vals):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                     f'{val:.3f}', ha='center', fontsize=9, fontweight='bold')
    # Target lines
    axes[1].axhline(0.85, color='#3498db', linestyle='--', alpha=0.6, linewidth=1.5,
                    label='AUC-ROC target (0.85)')
    axes[1].axhline(0.80, color='#e74c3c', linestyle='--', alpha=0.6, linewidth=1.5,
                    label='Recall target (0.80)')
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(OUTPUTS / '04a_cv_results.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("\n✅ Chart saved to outputs/04a_cv_results.png")
else:
    print("⚠️  Skipping cross-validation — data or xgboost not available.")

## Step 7: Train Final Model on Full Training Set

Cross-validation told us how the model performs across different data slices. Now we train the final model on the *entire* training set (all events 2010–2021) so it learns from as much data as possible before we evaluate it on the holdout.

> **Analogy:** Cross-validation is like practice games. This is the model we actually deploy.

In [ ]:
model = None

if X_train_resampled is not None and 'xgb' in dir():
    print("Training final model on full training set...")

    model = xgb.XGBClassifier(**XGB_PARAMS)
    model.fit(
        X_train_resampled,
        Y_train_resampled,
        eval_set=[(X_holdout, Y_holdout)],
        verbose=False
    )

    # ── Quick sanity check on training data ───────────────────────────────────
    train_probs = model.predict_proba(X_train)[:, 1]
    train_preds = (train_probs >= DECISION_THRESHOLD).astype(int)
    train_recall = recall_score(Y_train, train_preds, zero_division=0)
    train_auc    = roc_auc_score(Y_train, train_probs)

    print(f"✅ Final model trained.")
    print(f"   Trees used:         {model.n_estimators}")
    print(f"   Features:           {len(FEATURE_NAMES)}")
    print()
    print(f"Training set performance (sanity check):")
    print(f"  AUC-ROC:  {train_auc:.4f}")
    print(f"  Recall:   {train_recall:.4f}  (at threshold {DECISION_THRESHOLD})")
    print()
    print("ℹ️  Training performance is always higher than holdout/real-world.")
    print("   True evaluation is in Step 8 (holdout) and Notebook 05.")
else:
    print("⚠️  Cannot train — data or xgboost not available.")

## Step 8: Threshold Optimization

XGBoost outputs a probability score (0.0–1.0) for each ZCTA. We then apply a **decision threshold** to convert that probability into a binary prediction: flag this ZIP code, or don't.

The default threshold is 0.50. SustAInable uses 0.30.

This cell shows you exactly what happens to precision and recall at every possible threshold so you can see why 0.30 is the right choice — and update it if your data tells a different story.

> **The trade-off:** Lowering the threshold catches more true positives (higher recall) but also flags more false positives (lower precision). SustAInable accepts up to 2.5 false positives per true positive (minimum precision floor: 0.40).

In [ ]:
FINAL_THRESHOLD = DECISION_THRESHOLD

if model is not None and X_holdout is not None:
    holdout_probs = model.predict_proba(X_holdout)[:, 1]

    precisions, recalls, thresholds = precision_recall_curve(Y_holdout, holdout_probs)
    # Note: precision_recall_curve returns one extra value — align arrays
    thresholds_aligned = np.append(thresholds, 1.0)

    f1_scores  = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)
    pr_product = precisions * recalls  # maximize both simultaneously

    # ── Find optimal threshold: maximize recall subject to precision >= 0.40 ──
    valid_mask = precisions >= 0.40
    if valid_mask.any():
        best_idx       = np.argmax(recalls[valid_mask])
        FINAL_THRESHOLD= float(thresholds_aligned[valid_mask][best_idx])
        best_recall    = recalls[valid_mask][best_idx]
        best_precision = precisions[valid_mask][best_idx]
    else:
        # If we can't hit 0.40 precision, use the configured threshold
        FINAL_THRESHOLD = DECISION_THRESHOLD
        idx             = np.argmin(np.abs(thresholds_aligned - DECISION_THRESHOLD))
        best_recall     = recalls[idx]
        best_precision  = precisions[idx]
        print("⚠️  Could not reach precision ≥ 0.40 at any threshold.")
        print(f"   Using configured threshold: {DECISION_THRESHOLD}")

    print(f"Optimal threshold: {FINAL_THRESHOLD:.2f}")
    print(f"  At this threshold:")
    print(f"  Recall:    {best_recall:.4f}")
    print(f"  Precision: {best_precision:.4f}")

    # ── Visualization ──────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    fig.suptitle('SustAInable — Threshold Optimization', fontsize=13, fontweight='bold')

    # Precision-Recall curve
    axes[0].plot(recalls, precisions, color='#3498db', linewidth=2.5, label='PR Curve')
    axes[0].axvline(best_recall, color='#e74c3c', linestyle='--', linewidth=1.5,
                    label=f'Recall = {best_recall:.2f}')
    axes[0].axhline(0.40, color='#2ecc71', linestyle=':', linewidth=1.5,
                    label='Precision floor (0.40)')
    axes[0].scatter([best_recall], [best_precision], color='#e74c3c', s=120, zorder=5,
                    label=f'Selected threshold ({FINAL_THRESHOLD:.2f})')
    axes[0].set_xlabel('Recall (sensitivity)')
    axes[0].set_ylabel('Precision')
    axes[0].set_title('Precision-Recall Curve (Holdout Set)')
    axes[0].legend(fontsize=9)
    axes[0].set_xlim(0, 1.05)
    axes[0].set_ylim(0, 1.05)

    # Metrics vs threshold
    plot_range = (thresholds_aligned >= 0.10) & (thresholds_aligned <= 0.90)
    axes[1].plot(thresholds_aligned[plot_range], recalls[plot_range],
                 color='#e74c3c', linewidth=2, label='Recall')
    axes[1].plot(thresholds_aligned[plot_range], precisions[plot_range],
                 color='#2ecc71', linewidth=2, label='Precision')
    axes[1].plot(thresholds_aligned[plot_range], f1_scores[plot_range],
                 color='#9b59b6', linewidth=2, linestyle='--', label='F1')
    axes[1].axvline(FINAL_THRESHOLD, color='#e74c3c', linestyle=':', linewidth=2,
                    label=f'Selected ({FINAL_THRESHOLD:.2f})')
    axes[1].axvline(0.50, color='#95a5a6', linestyle=':', linewidth=1.5,
                    label='Default (0.50)', alpha=0.7)
    axes[1].axhline(0.80, color='#e74c3c', linestyle=':', alpha=0.4)
    axes[1].axhline(0.40, color='#2ecc71', linestyle=':', alpha=0.4)
    axes[1].set_xlabel('Decision Threshold')
    axes[1].set_ylabel('Score')
    axes[1].set_title('Precision / Recall / F1 vs. Threshold')
    axes[1].legend(fontsize=9)
    axes[1].set_xlim(0.10, 0.90)
    axes[1].set_ylim(0, 1.05)

    plt.tight_layout()
    plt.savefig(OUTPUTS / '04b_threshold_curve.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("\n✅ Chart saved to outputs/04b_threshold_curve.png")
else:
    print("⚠️  Skipping threshold optimization — model not available.")

## Step 9: Holdout Set Evaluation

This is the most important number in the notebook. The holdout set (2022–2024 events) has never been touched during training. Performance here is our best estimate of how the model will perform on future heat events.

The three holdout events — 2022 Pacific Northwest, 2023 Texas record heat, and 2024 Phoenix multi-week event — were deliberately chosen as stress tests because they are among the most severe documented US heat events on record.

In [ ]:
if model is not None and X_holdout is not None:
    holdout_probs = model.predict_proba(X_holdout)[:, 1]
    holdout_preds = (holdout_probs >= FINAL_THRESHOLD).astype(int)

    auc_roc   = roc_auc_score(Y_holdout, holdout_probs)
    pr_auc    = average_precision_score(Y_holdout, holdout_probs)
    recall    = recall_score(Y_holdout, holdout_preds, zero_division=0)
    precision = precision_score(Y_holdout, holdout_preds, zero_division=0)
    f1        = f1_score(Y_holdout, holdout_preds, zero_division=0)
    cm        = confusion_matrix(Y_holdout, holdout_preds)

    tn, fp, fn, tp = cm.ravel() if cm.shape == (2,2) else (cm[0,0], 0, 0, 0)

    print("=" * 58)
    print(f"HOLDOUT SET PERFORMANCE  (threshold = {FINAL_THRESHOLD:.2f})")
    print("=" * 58)

    results = [
        ("AUC-ROC",   auc_roc,   0.85, "Discrimination ability"),
        ("PR-AUC",    pr_auc,    None, "Precision-recall area (imbalanced data)"),
        ("Recall",    recall,    0.80, "% elevated ZCTAs we catch  ← primary metric"),
        ("Precision", precision, 0.40, "% of flags that are correct"),
        ("F1 Score",  f1,        None, "Harmonic mean of recall and precision"),
    ]

    for name, val, target, note in results:
        if target:
            status = "✅" if val >= target else "⚠️ "
            target_str = f"  [target ≥ {target}]"
        else:
            status = "  "
            target_str = ""
        print(f"  {status} {name:<12} {val:.4f}{target_str}  — {note}")

    print()
    print("Confusion Matrix (holdout):")
    print(f"  True Negatives  (correct non-flags):  {tn:>6,}")
    print(f"  False Positives (unnecessary flags):  {fp:>6,}  ← acceptable cost")
    print(f"  False Negatives (missed elevations):  {fn:>6,}  ← the costly error")
    print(f"  True Positives  (correct flags):      {tp:>6,}")

    if fn > 0 and tp > 0:
        miss_rate = fn / (fn + tp)
        print(f"\n  Miss rate: {miss_rate:.2%} of elevated ZCTAs not flagged")

    # ── Baseline comparison ────────────────────────────────────────────────────
    # CDC HHI standalone AUC benchmark from published literature
    HHI_BASELINE_AUC = 0.73
    improvement = ((auc_roc - HHI_BASELINE_AUC) / HHI_BASELINE_AUC) * 100
    print()
    print(f"vs. CDC HHI baseline (AUC ≈ {HHI_BASELINE_AUC}):")
    if auc_roc > HHI_BASELINE_AUC:
        print(f"  ✅ SustAInable outperforms baseline by {improvement:.1f}%")
    else:
        print(f"  ⚠️  Below baseline — review feature engineering and class weighting")
else:
    print("⚠️  Holdout evaluation skipped — model not available.")

## Step 10: Feature Importance Preview

XGBoost tracks how often each feature was used to make split decisions across all trees. This isn't the same as SHAP values (that's Notebook 06) — it's a quick diagnostic to check that the model is using the features we expect and not leaning too heavily on any single one.

> **Red flag:** If one feature accounts for >40% of importance, the model may be overfitting to it. We want to see a spread across multiple vulnerability indicators.

In [ ]:
if model is not None:
    importance_df = pd.DataFrame({
        'feature': FEATURE_NAMES,
        'importance_weight': model.feature_importances_
    }).sort_values('importance_weight', ascending=False).reset_index(drop=True)

    importance_df['importance_pct'] = (
        importance_df['importance_weight'] /
        importance_df['importance_weight'].sum() * 100
    ).round(2)

    top_n = min(20, len(importance_df))
    top_features = importance_df.head(top_n)

    print(f"Top {top_n} Features by Importance (gain):")
    display(top_features)

    # Check for concentration
    top3_pct = importance_df.head(3)['importance_pct'].sum()
    if top3_pct > 60:
        print(f"\n⚠️  Top 3 features account for {top3_pct:.1f}% of importance.")
        print("   Consider reviewing feature correlation and engineering.")
    else:
        print(f"\n✅ Top 3 features: {top3_pct:.1f}% of importance — healthy spread.")

    # ── Plot ───────────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(10, max(6, top_n * 0.4)))
    colors = ['#e74c3c' if i < 3 else '#3498db' for i in range(top_n)]
    bars = ax.barh(
        top_features['feature'][::-1],
        top_features['importance_pct'][::-1],
        color=colors[::-1], edgecolor='white', alpha=0.85
    )
    ax.set_xlabel('Feature Importance (%)')
    ax.set_title(f'SustAInable — Top {top_n} Feature Importances\n(gain; red = top 3)',
                 fontsize=12)
    for bar, val in zip(bars, top_features['importance_pct'][::-1]):
        ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                f'{val:.1f}%', va='center', fontsize=8)
    plt.tight_layout()
    plt.savefig(OUTPUTS / '04c_feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Chart saved to outputs/04c_feature_importance.png")
else:
    print("⚠️  Feature importance skipped — model not available.")

## Step 11: Save Model and Metadata

Saves two files:

1. **`sustainable_model.json`** — the trained XGBoost model in portable JSON format. This is the file you would load in a deployment pipeline to score new ZCTAs.

2. **`sustainable_model_metadata.json`** — everything needed to reproduce, audit, and deploy this model: the threshold, feature list, training config, performance metrics, and a proxy data warning if applicable.

In [ ]:
if model is not None:
    # ── Save XGBoost model ─────────────────────────────────────────────────────
    model.save_model(str(MODEL_PATH))
    print(f"✅ Model saved: {MODEL_PATH}")

    # ── Build metadata ─────────────────────────────────────────────────────────
    holdout_probs_final = model.predict_proba(X_holdout)[:, 1]
    holdout_preds_final = (holdout_probs_final >= FINAL_THRESHOLD).astype(int)

    metadata = {
        "project":          "SustAInable — Neighborhood Heat Illness Risk Prediction",
        "author":           "Nico Meyering, MPA",
        "institution":      "Equitech Futures Civic Tech Institute, CTI 2026",
        "model_type":       "XGBoost Binary Classifier",
        "label_mode":       "proxy" if IS_PROXY else "real_nssp",
        "proxy_warning":    (
            "LABELS ARE STRUCTURALLY DERIVED PROXIES. "
            "Do not use for real-world deployment. "
            "Replace with NSSP data via BioSense DUA before production use."
        ) if IS_PROXY else None,
        "decision_threshold": FINAL_THRESHOLD,
        "n_features":       len(FEATURE_NAMES),
        "feature_names":    FEATURE_NAMES,
        "training_events":  "2010-2021",
        "holdout_events":   "2022-2024",
        "xgb_params":       {k: v for k, v in XGB_PARAMS.items()
                             if k not in ['eval_metric']},
        "holdout_performance": {
            "auc_roc":   round(float(roc_auc_score(Y_holdout, holdout_probs_final)), 4),
            "pr_auc":    round(float(average_precision_score(Y_holdout, holdout_probs_final)), 4),
            "recall":    round(float(recall_score(Y_holdout, holdout_preds_final, zero_division=0)), 4),
            "precision": round(float(precision_score(Y_holdout, holdout_preds_final, zero_division=0)), 4),
            "f1":        round(float(f1_score(Y_holdout, holdout_preds_final, zero_division=0)), 4),
        },
        "cv_performance":   cv_results_summary if cv_results_summary else "not run",
        "asymmetric_cost_design": {
            "false_negative_cost": "Disabled/elderly residents experience preventable heat illness — NOT recoverable",
            "false_positive_cost": "Cooling resources deployed unnecessarily (~$2k-8k per ZCTA) — recoverable",
            "design_decision":     "Threshold lowered to maximize recall subject to precision >= 0.40",
        },
        "risk_tiers": {
            "high":     ">= 0.60",
            "elevated": "0.30 - 0.59",
            "baseline": "< 0.30"
        }
    }

    with open(METADATA_PATH, "w") as f:
        json.dump(metadata, f, indent=2)
    print(f"✅ Metadata saved: {METADATA_PATH}")

    # Summary
    print()
    print("Files written:")
    for p in [MODEL_PATH, METADATA_PATH]:
        size_kb = p.stat().st_size / 1024
        print(f"  {p.name}  ({size_kb:.1f} KB)")
else:
    print("⚠️  Nothing to save — model not trained.")

---

## ✅ What You Just Built

A trained XGBoost classifier that predicts which ZIP codes will experience elevated heat illness during an approaching extreme heat event.

The model was validated on three major historical heat events it never saw during training: the 2022 Pacific Northwest dome, the 2023 Texas record heat, and the 2024 Phoenix multi-week event.

---

## 🔜 What Comes Next

| Notebook | Purpose |
|---|---|
| `05_model_evaluation.ipynb` | Full evaluation: ROC curves, calibration, per-event analysis, HHI comparison |
| `06_shap_explainability.ipynb` | SHAP values — why each ZIP code was flagged |

---

## ⚠️ If You See Proxy Label Warnings

That's expected at this stage. Everything built here is the right pipeline — you're one NSSP dataset away from training on real data.

**NSSP access request:** https://www.cdc.gov/nssp/php/onboarding/index.html

---

*SustAInable — Neighborhood Heat Illness Risk Prediction*  
*Equitech Futures Civic Tech Institute, CTI 2026*